# Split a van's pings into trips

Two delivery vans send a GPS ping every few minutes while they drive, and stop
sending while they are parked. The server stores the pings in the order they arrive. You
will turn this log into trips, numbered per van, and then into a table with one row per
trip, all without a loop over the rows.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np
import pandas as pd

# GPS pings from two delivery vans, in the order the server received them.
# minute counts the minutes since midnight.
log = pd.DataFrame({
    "van":    ["V1", "V1", "V2", "V1", "V1", "V1", "V2", "V2", "V1", "V1", "V2", "V2"],
    "minute": [540, 600, 655, 480, 486, 510, 612, 605, 495, 548, 640, 700],
})
print(log)

## Part 1 (coding task): Number each van's trips

A van that sends no ping for more than 15 minutes has parked, and its next ping starts a
new trip. Write `add_trip_numbers(log, gap=15)`, which returns a new table with

- the rows sorted by `van` and then by `minute`, with the index running 0, 1, 2, ....
- a column `trip` that is 1 for every van's first trip and goes up by 1 each time the
  time since **the same van's** previous ping is more than `gap` minutes.

A gap of exactly `gap` minutes does not start a trip. Leave `log` itself unchanged, and use
column operations, not a loop over rows.

A hint: `df.groupby("van")["minute"].diff()` gives each ping's time since the previous
ping of the same van, and NaN for each van's first ping.

In [ ]:
def add_trip_numbers(log, gap=15):
    """The pings sorted by van and minute, with each van's trips numbered from 1."""
    # TODO
    raise NotImplementedError

In [ ]:
before = log.copy()
trips = add_trip_numbers(log)
print(trips)
assert log.equals(before), "add_trip_numbers changed log: work on a sorted copy"
assert "trip" in trips.columns, "the result has no 'trip' column"
assert list(trips.index) == list(range(len(log))), "reset the index so it runs 0, 1, 2, ..."
assert list(trips["van"]) == ["V1"] * 7 + ["V2"] * 5, "sort by van first, then by minute"
assert list(trips["minute"]) == [480, 486, 495, 510, 540, 548, 600, 605, 612, 640, 655, 700], \
    "within each van, sort the pings by minute"
v1 = list(trips.loc[trips["van"] == "V1", "trip"])
v2 = list(trips.loc[trips["van"] == "V2", "trip"])
assert v1[0] == 1, "number each van's trips from 1: its first ping, whose gap is NaN, starts trip 1"
assert v1[3] == 1, "V1's ping at 510 comes exactly 15 minutes after the one at 495: a gap of exactly 15 does not start a trip"
assert v2[0] == 1, f"V2's first ping should start its trip 1, got {v2[0]}: compute the gap within each van and number each van's trips from 1"
assert v1 == [1, 1, 1, 1, 2, 2, 3], f"V1's trips should be [1, 1, 1, 1, 2, 2, 3], got {v1}"
assert v2 == [1, 1, 2, 2, 3], f"V2's trips should be [1, 1, 2, 2, 3], got {v2}"
loose = add_trip_numbers(log, gap=40)
assert list(loose["trip"]) == [1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 2], "the gap argument should set the threshold"
print("Part 1 passed")

## Part 2 (coding task): One row per trip

Write `trip_table(trips)`, which takes a table like the one `add_trip_numbers` returns and
gives one row per trip, with the ordinary columns `van`, `trip`, `start` (the first ping's
minute), `end` (the last ping's minute), `pings` (how many pings) and `minutes`
(`end - start`). Order the rows by van and then by trip, with the index running 0, 1, 2, ...
A trip of one ping has 0 minutes.

In [ ]:
def trip_table(trips):
    """One row per (van, trip): start, end, number of pings and length in minutes."""
    # TODO
    raise NotImplementedError

In [ ]:
# The numbered pings, written out here so this check does not depend on Part 1.
numbered = pd.DataFrame({
    "van":    ["V1"] * 7 + ["V2"] * 5,
    "minute": [480, 486, 495, 510, 540, 548, 600, 605, 612, 640, 655, 700],
    "trip":   [1, 1, 1, 1, 2, 2, 3, 1, 1, 2, 2, 3],
})
table = trip_table(numbered)
print(table)
assert len(table) == 6, \
    f"expected one row per trip of each van, 6 in all, got {len(table)}: trip numbers restart for every van, so group by van and trip together"
for col in ["van", "trip", "start", "end", "pings", "minutes"]:
    assert col in table.columns, f"the table has no {col!r} column: keep van and trip as columns, not as the index"
assert list(table.index) == list(range(6)), "the index should run 0, 1, 2, ..."
assert list(zip(table["van"], table["trip"])) == [("V1", 1), ("V1", 2), ("V1", 3), ("V2", 1), ("V2", 2), ("V2", 3)], \
    "order the rows by van, then by trip"
assert list(table["start"]) == [480, 540, 600, 605, 640, 700], f"start is wrong: {list(table['start'])}"
assert list(table["end"]) == [510, 548, 600, 612, 655, 700], f"end is wrong: {list(table['end'])}"
assert list(table["pings"]) == [4, 2, 1, 2, 2, 1], f"pings is wrong: {list(table['pings'])}"
assert list(table["minutes"]) == [30, 8, 0, 7, 15, 0], f"minutes should be end minus start: {list(table['minutes'])}"
print("Part 2 passed")

## Solution

### Part 1: Number each van's trips

In [ ]:
def add_trip_numbers(log, gap=15):
    """The pings sorted by van and minute, with each van's trips numbered from 1."""
    out = log.sort_values(["van", "minute"]).reset_index(drop=True)
    since_last = out.groupby("van")["minute"].diff()
    starts = since_last.isna() | (since_last > gap)
    out["trip"] = starts.groupby(out["van"]).cumsum().astype(int)
    return out

**Why it works.** After sorting, `groupby("van")["minute"].diff()` subtracts each ping's
previous ping **of the same van**, so V2's first ping gets NaN instead of $605 - 600 = 5$, the
gap to V1's last ping, which would wrongly glue it onto a V1 trip. A ping starts a trip when
that difference is missing or above the gap. `cumsum` within each van then counts the starts
so far: V1's starts are at 480, 540 and 600, so its pings get 1, 1, 1, 1, 2, 2, 3. The test is
`> gap`, so the ping at 510, exactly 15 minutes after 495, stays in trip 1.

### Part 2: One row per trip

In [ ]:
def trip_table(trips):
    """One row per (van, trip): start, end, number of pings and length in minutes."""
    table = trips.groupby(["van", "trip"], as_index=False).agg(
        start=("minute", "min"),
        end=("minute", "max"),
        pings=("minute", "size"),
    )
    table["minutes"] = table["end"] - table["start"]
    return table

**Why it works.** Grouping by both keys makes one group per trip of each van, which matters
because trip numbers restart for every van: grouping by `trip` alone would merge V1's trip 1
with V2's trip 1. `as_index=False` keeps `van` and `trip` as ordinary columns, and the groups
come out sorted by their keys. `min`, `max` and `size` of `minute` give the start, the end
and the number of pings, and the length is ordinary column arithmetic on the result.